# LG-CoTrain: Quick-Stop with Optuna-Tuned Hyperparameters

This notebook repeats the **quick comparison** of all 6 stopping strategies from
[notebook 04](04_alternative_stopping_strategies.ipynb), but replaces the fixed default
hyperparameters with **per-event Optuna-tuned** hyperparameters found by the per-experiment
tuner (`optuna_per_experiment.py`).

| Strategy | Key Idea |
|---|---|
| `baseline` | Original: stop when ensemble macro-F1 plateaus for `patience` epochs |
| `no_early_stopping` | Run all `finetune_max_epochs`; restore best-ever checkpoint (upper bound) |
| `per_class_patience` | Stop only when **every** class F1 has individually plateaued |
| `weighted_macro_f1` | Weight rare classes more in the stopping metric |
| `balanced_dev` | Resample dev set to equal class sizes for the stopping signal |
| `scaled_threshold` | Require a larger improvement delta for highly imbalanced events |

**Scope**: budget=50, seed=1 across all 10 events (same as notebook 04).

**Total experiments**: 6 strategies x 10 events x 1 budget x 1 seed = **60 runs**

**Key difference from notebook 04**: Each event uses its own optimal `lr`, `batch_size`,
`cotrain_epochs`, `finetune_patience`, `weight_decay`, and `warmup_ratio` from Optuna tuning,
instead of the fixed defaults (lr=2e-5, batch_size=32, etc.).

Results are stored in `results/{PSEUDO_LABEL_SOURCE}/optuna-quick-stop/{strategy}/`
(parallel to notebook 04's `results/.../quick-stop/{strategy}/`).

In [1]:
import json
import sys
import time
from pathlib import Path


def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(
        f"Cannot find repo root: no ancestor directory contains '{marker}/'. "
        "Run the notebook from inside the repository."
    )


repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np

from lg_cotrain.run_all import run_all_experiments
from lg_cotrain.parallel import run_experiments_parallel
from lg_cotrain.optuna_per_experiment import load_best_params


class ProgressTracker:
    """Track global progress across all strategies x events x budgets x seeds."""

    def __init__(self, total: int, already_done: int, start_time: float):
        self.total = total
        self.done = already_done
        self.start_time = start_time

    def update(self, event, budget, seed_set, status):
        self.done += 1
        elapsed = time.time() - self.start_time
        pct = 100.0 * self.done / self.total if self.total else 0
        elapsed_h = elapsed / 3600
        remaining = self.total - self.done
        eta_h = (elapsed / self.done) * remaining / 3600 if self.done > 0 else 0
        print(
            f"  [PROGRESS] {self.done}/{self.total} ({pct:.1f}%)"
            f"  |  Elapsed: {elapsed_h:.2f}h  |  ETA: {eta_h:.2f}h  |  {status}"
        )


print(f"Repo root: {repo_root}")

Repo root: D:\Workspace\Co-Training


In [2]:
# ---- Configuration ----

PSEUDO_LABEL_SOURCE = "gpt-4o"

# Quick-run scope: 50 labels/class, seed set 1 only
RUN_BUDGETS = [50]
RUN_SEEDS   = [1]

# Number of GPUs for parallel execution (1 = sequential fallback)
NUM_GPUS = 2

# Optuna trial count to load (None = latest available)
N_TRIALS = 10

STRATEGIES = [
    "baseline",
    "no_early_stopping",
    "per_class_patience",
    "weighted_macro_f1",
    "balanced_dev",
    "scaled_threshold",
]

DATA_ROOT = str(repo_root / "data")

# Discover all events
TARGET_EVENTS = sorted(
    p.name for p in (Path(DATA_ROOT) / "original").iterdir() if p.is_dir()
)

# Each strategy gets its own results sub-folder under model/optuna-quick-stop/
STRATEGY_RESULTS_ROOTS = {
    s: str(repo_root / "results" / PSEUDO_LABEL_SOURCE / "optuna-quick-stop" / s)
    for s in STRATEGIES
}

# ---- Load Optuna best params ----
OPTUNA_DIR = str(repo_root / "results" / "optuna" / "per_experiment")
optuna_results = load_best_params(
    storage_dir=OPTUNA_DIR,
    budgets=RUN_BUDGETS,
    seed_sets=RUN_SEEDS,
    n_trials=N_TRIALS,
)

# Default hyperparams (fallback when Optuna results are missing)
DEFAULT_PARAMS = {
    "lr": 2e-5,
    "batch_size": 32,
    "cotrain_epochs": 10,
    "finetune_patience": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
}

# Build per-event param lookup
event_params = {}  # event -> dict with the 6 hyperparams
for event in TARGET_EVENTS:
    key = (event, RUN_BUDGETS[0], RUN_SEEDS[0])
    if key in optuna_results:
        event_params[event] = optuna_results[key]["best_params"]
    else:
        event_params[event] = None  # will use defaults

# ---- Print summary ----
total_runs = len(STRATEGIES) * len(TARGET_EVENTS) * len(RUN_BUDGETS) * len(RUN_SEEDS)
print(f"Strategies : {STRATEGIES}")
print(f"Events     : {TARGET_EVENTS}")
print(f"Budget     : {RUN_BUDGETS}  |  Seed sets: {RUN_SEEDS}")
print(f"Total runs : {total_runs}  |  GPUs: {NUM_GPUS}")
print(f"Optuna dir : {OPTUNA_DIR}  (n_trials={N_TRIALS})")
print()

print(f"{'Event':<35} {'lr':>10} {'batch':>6} {'co_ep':>6} {'ft_pat':>7} {'w_dec':>8} {'warmup':>8}  Source")
print("-" * 100)
for event in TARGET_EVENTS:
    p = event_params[event] if event_params[event] else DEFAULT_PARAMS
    src = "optuna" if event_params[event] else "DEFAULT"
    print(
        f"{event:<35} {p['lr']:>10.6f} {p['batch_size']:>6} {p['cotrain_epochs']:>6} "
        f"{p['finetune_patience']:>7} {p['weight_decay']:>8.4f} {p['warmup_ratio']:>8.4f}  {src}"
    )

missing = [e for e in TARGET_EVENTS if event_params[e] is None]
if missing:
    print(f"\nWARNING: {len(missing)} event(s) missing Optuna results, using defaults: {missing}")

print()
for s, r in STRATEGY_RESULTS_ROOTS.items():
    print(f"  {s:<25} -> {r}")

Strategies : ['baseline', 'no_early_stopping', 'per_class_patience', 'weighted_macro_f1', 'balanced_dev', 'scaled_threshold']
Events     : ['california_wildfires_2018', 'canada_wildfires_2016', 'cyclone_idai_2019', 'hurricane_dorian_2019', 'hurricane_florence_2018', 'hurricane_harvey_2017', 'hurricane_irma_2017', 'hurricane_maria_2017', 'kaikoura_earthquake_2016', 'kerala_floods_2018']
Budget     : [50]  |  Seed sets: [1]
Total runs : 60  |  GPUs: 2
Optuna dir : D:\Workspace\Co-Training\results\optuna\per_experiment  (n_trials=10)

Event                                       lr  batch  co_ep  ft_pat    w_dec   warmup  Source
----------------------------------------------------------------------------------------------------
california_wildfires_2018             0.000032     16     17       6   0.0581   0.0452  optuna
canada_wildfires_2016                 0.000014     16     17       9   0.0839   0.2243  optuna
cyclone_idai_2019                     0.000116     32     14      10   0.081

## Running Experiments

Each event uses its own Optuna-tuned hyperparameters.
If the cell crashes or is interrupted, re-run it -- existing `metrics.json` files
are automatically skipped.

In [3]:
total_experiments = len(STRATEGIES) * len(TARGET_EVENTS) * len(RUN_BUDGETS) * len(RUN_SEEDS)

print(f"Total experiments : {total_experiments}")
print(f"Execution mode    : {'parallel (' + str(NUM_GPUS) + ' GPUs)' if NUM_GPUS > 1 else 'sequential'}")
print()

overall_start = time.time()
tracker = ProgressTracker(total_experiments, 0, overall_start)
all_strategy_results = {}  # strategy -> event -> list[result_dict]

for strategy in STRATEGIES:
    results_root = STRATEGY_RESULTS_ROOTS[strategy]
    strat_start = time.time()
    print(f"\n{'=' * 65}")
    print(f"Strategy: {strategy}")
    print(f"{'=' * 65}")
    all_strategy_results[strategy] = {}

    # Build list of pending experiment configs, skipping completed ones
    pending_configs = []   # list of LGCoTrainConfig kwarg dicts
    pending_events = []    # corresponding event names (for result mapping)
    skipped_results = {}   # event -> result dict (loaded from disk)

    for event in TARGET_EVENTS:
        budget = RUN_BUDGETS[0]
        seed_set = RUN_SEEDS[0]
        metrics_path = Path(results_root) / event / f"{budget}_set{seed_set}" / "metrics.json"

        if metrics_path.exists():
            # Resume: load existing result
            with open(metrics_path) as f:
                skipped_results[event] = json.load(f)
            print(f"  {event} budget={budget}, seed={seed_set} -- SKIPPED (exists)")
            tracker.update(event, budget, seed_set, "skipped")
            continue

        # Build config dict with per-event Optuna hyperparams
        p = event_params[event] if event_params[event] else DEFAULT_PARAMS
        pending_configs.append(dict(
            event=event,
            budget=budget,
            seed_set=seed_set,
            pseudo_label_source=PSEUDO_LABEL_SOURCE,
            stopping_strategy=strategy,
            lr=p["lr"],
            batch_size=p["batch_size"],
            cotrain_epochs=p["cotrain_epochs"],
            finetune_patience=p["finetune_patience"],
            weight_decay=p["weight_decay"],
            warmup_ratio=p["warmup_ratio"],
            data_root=DATA_ROOT,
            results_root=results_root,
        ))
        pending_events.append(event)

    # Dispatch pending experiments
    if pending_configs and NUM_GPUS > 1:
        print(f"\n  Dispatching {len(pending_configs)} experiments across {NUM_GPUS} GPUs...")
        outcomes = run_experiments_parallel(
            pending_configs,
            num_gpus=NUM_GPUS,
            on_experiment_done=tracker.update,
        )
        for event, outcome in zip(pending_events, outcomes):
            if outcome["status"] == "done" and outcome["result"] is not None:
                all_strategy_results[strategy][event] = [outcome["result"]]
            else:
                print(f"  WARNING: {event} failed")
                all_strategy_results[strategy][event] = []

    elif pending_configs:
        # Sequential fallback (NUM_GPUS == 1)
        for cfg, event in zip(pending_configs, pending_events):
            results = run_all_experiments(
                event,
                budgets=RUN_BUDGETS,
                seed_sets=RUN_SEEDS,
                pseudo_label_source=PSEUDO_LABEL_SOURCE,
                stopping_strategy=strategy,
                lr=cfg["lr"],
                batch_size=cfg["batch_size"],
                cotrain_epochs=cfg["cotrain_epochs"],
                finetune_patience=cfg["finetune_patience"],
                weight_decay=cfg["weight_decay"],
                warmup_ratio=cfg["warmup_ratio"],
                data_root=DATA_ROOT,
                results_root=results_root,
                _on_experiment_done=tracker.update,
            )
            all_strategy_results[strategy][event] = results

    # Merge skipped results
    for event, result in skipped_results.items():
        all_strategy_results[strategy][event] = [result]

    strat_elapsed = time.time() - strat_start
    print(f"\n  Strategy '{strategy}' done in {strat_elapsed / 3600:.2f}h ({strat_elapsed / 60:.1f}min)")

total_elapsed = time.time() - overall_start
print(f"\n{'=' * 65}")
print(f"All experiments complete in {total_elapsed / 3600:.2f}h ({total_elapsed / 60:.1f}min)")

Total experiments : 60
Execution mode    : parallel (2 GPUs)


Strategy: baseline
  california_wildfires_2018 budget=50, seed=1 -- SKIPPED (exists)
  [PROGRESS] 1/60 (1.7%)  |  Elapsed: 0.00h  |  ETA: 0.00h  |  skipped
  canada_wildfires_2016 budget=50, seed=1 -- SKIPPED (exists)
  [PROGRESS] 2/60 (3.3%)  |  Elapsed: 0.00h  |  ETA: 0.00h  |  skipped
  cyclone_idai_2019 budget=50, seed=1 -- SKIPPED (exists)
  [PROGRESS] 3/60 (5.0%)  |  Elapsed: 0.00h  |  ETA: 0.00h  |  skipped
  hurricane_dorian_2019 budget=50, seed=1 -- SKIPPED (exists)
  [PROGRESS] 4/60 (6.7%)  |  Elapsed: 0.00h  |  ETA: 0.00h  |  skipped
  hurricane_florence_2018 budget=50, seed=1 -- SKIPPED (exists)
  [PROGRESS] 5/60 (8.3%)  |  Elapsed: 0.00h  |  ETA: 0.00h  |  skipped
  hurricane_harvey_2017 budget=50, seed=1 -- SKIPPED (exists)
  [PROGRESS] 6/60 (10.0%)  |  Elapsed: 0.00h  |  ETA: 0.00h  |  skipped
  hurricane_irma_2017 budget=50, seed=1 -- SKIPPED (exists)
  [PROGRESS] 7/60 (11.7%)  |  Elapsed: 0.00h  |  ETA: 0.0

In [4]:
# Load any results that already existed (re-run safe)
for strategy in STRATEGIES:
    results_root = Path(STRATEGY_RESULTS_ROOTS[strategy])
    if strategy not in all_strategy_results:
        all_strategy_results[strategy] = {}

    for event in TARGET_EVENTS:
        if event in all_strategy_results[strategy]:
            continue
        results = []
        for budget in RUN_BUDGETS:
            for seed_set in RUN_SEEDS:
                path = results_root / event / f"{budget}_set{seed_set}" / "metrics.json"
                if path.exists():
                    with open(path) as f:
                        results.append(json.load(f))
        if results:
            all_strategy_results[strategy][event] = results

# Build flat lookup: lookup[strategy][event] -> result dict (or None)
# Since budget=50, seed=1 only, each (strategy, event) has at most one result.
lookup = {}
for strategy in STRATEGIES:
    lookup[strategy] = {}
    for event in TARGET_EVENTS:
        results = all_strategy_results.get(strategy, {}).get(event, [])
        # Pick the single result for budget=50, seed=1
        match = next(
            (r for r in results if r and r.get("budget") == 50 and r.get("seed_set") == 1),
            None,
        )
        lookup[strategy][event] = match

print("Results available (budget=50, seed=1, Optuna hyperparams):")
print(f"{'Strategy':<26}" + "".join(f" {e[:12]:<13}" for e in TARGET_EVENTS))
print("-" * (26 + 14 * len(TARGET_EVENTS)))
for strategy in STRATEGIES:
    row = f"{strategy:<26}"
    for event in TARGET_EVENTS:
        row += " OK          " if lookup[strategy][event] else " --          "
    print(row)

NameError: name 'all_strategy_results' is not defined

In [ ]:
# Summary table: strategies (rows) x events (columns), value = macro-F1
# Plus a delta-from-baseline table.

col_w = 10
event_labels = [e.replace("_", " ") for e in TARGET_EVENTS]
short_labels  = [" ".join(w[:4] for w in e.split("_")) for e in TARGET_EVENTS]

print("Macro-F1  (budget=50, seed=1, Optuna hyperparams)")
print(f"{'Strategy':<26}" + "".join(f" {sl:<{col_w}}" for sl in short_labels) + "  Mean")
print("-" * (26 + (col_w + 1) * len(TARGET_EVENTS) + 6))

baseline_row = {}
for strategy in STRATEGIES:
    row = f"{strategy:<26}"
    vals = []
    for event in TARGET_EVENTS:
        r = lookup[strategy][event]
        f1 = r["test_macro_f1"] if r else None
        vals.append(f1)
        row += f" {f1:.4f}   " if f1 is not None else f" {'N/A':<{col_w}}"
    valid = [v for v in vals if v is not None]
    row += f"  {sum(valid)/len(valid):.4f}" if valid else "  N/A"
    print(row)
    if strategy == "baseline":
        baseline_row = {e: v for e, v in zip(TARGET_EVENTS, vals)}

print()
print("Delta vs baseline  (+) = better:")
print(f"{'Strategy':<26}" + "".join(f" {sl:<{col_w}}" for sl in short_labels) + "  Mean delta")
print("-" * (26 + (col_w + 1) * len(TARGET_EVENTS) + 12))

for strategy in STRATEGIES:
    if strategy == "baseline":
        continue
    row = f"{strategy:<26}"
    deltas = []
    for event in TARGET_EVENTS:
        r = lookup[strategy][event]
        f1   = r["test_macro_f1"] if r else None
        base = baseline_row.get(event)
        if f1 is not None and base is not None:
            d = f1 - base
            deltas.append(d)
            sign = "+" if d >= 0 else ""
            row += f" {sign}{d:.4f}  "
        else:
            row += f" {'N/A':<{col_w}}"
    row += f"  {'+' if sum(deltas)/len(deltas)>=0 else ''}{sum(deltas)/len(deltas):.4f}" if deltas else "  N/A"
    print(row)

In [ ]:
# Grouped bar chart: one group per event, one bar per strategy
# All at budget=50, seed=1, Optuna hyperparams

n_events     = len(TARGET_EVENTS)
n_strategies = len(STRATEGIES)
bar_width    = 0.8 / n_strategies
colors       = plt.cm.tab10(np.linspace(0, 1, n_strategies))
x            = np.arange(n_events)

fig, ax = plt.subplots(figsize=(max(14, n_events * 1.4), 5))

for i, (strategy, color) in enumerate(zip(STRATEGIES, colors)):
    f1s = [
        lookup[strategy][event]["test_macro_f1"]
        if lookup[strategy][event] else 0
        for event in TARGET_EVENTS
    ]
    offset = (i - n_strategies / 2 + 0.5) * bar_width
    ax.bar(x + offset, f1s, bar_width * 0.9, label=strategy, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(
    [e.replace("_", "\n") for e in TARGET_EVENTS],
    fontsize=8,
)
ax.set_ylabel("Test Macro-F1")
ax.set_ylim(0, 1)
ax.set_title(
    f"Stopping Strategy Comparison (Optuna Hyperparams) -- Budget=50, Seed=1\n"
    f"(pseudo-labels: {PSEUDO_LABEL_SOURCE})",
    fontsize=11,
)
ax.legend(loc="upper right", fontsize=8, framealpha=0.8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 heatmap for each event
# Rows = strategies, Columns = classes, Value = F1 at budget=50, seed=1

from lg_cotrain.data_loading import CLASS_LABELS

for event in TARGET_EVENTS:
    strategies_with_data = [
        s for s in STRATEGIES
        if lookup[s][event] and "test_per_class_f1" in lookup[s][event]
    ]
    if not strategies_with_data:
        print(f"No per-class data for {event}, skipping.")
        continue

    # Determine number of classes for this event from actual data length
    n_classes = len(lookup[strategies_with_data[0]][event]["test_per_class_f1"])
    class_labels = CLASS_LABELS[:n_classes] if n_classes <= len(CLASS_LABELS) else [
        f"class_{i}" for i in range(n_classes)
    ]

    data = np.array([
        lookup[s][event]["test_per_class_f1"]
        for s in strategies_with_data
    ])  # shape: (n_strategies, n_classes)

    fig, ax = plt.subplots(
        figsize=(max(9, n_classes * 0.75), len(strategies_with_data) * 0.65 + 1.8)
    )
    im = ax.imshow(data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(strategies_with_data)))
    ax.set_yticklabels(strategies_with_data, fontsize=9)
    ax.set_title(
        f"{event}  |  Budget=50, Seed=1 (Optuna)  |  Per-class F1 by strategy",
        fontsize=10,
    )

    for i in range(len(strategies_with_data)):
        for j in range(n_classes):
            val = data[i, j]
            color = "black" if 0.25 < val < 0.75 else "white"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)

    fig.colorbar(im, ax=ax, label="F1 Score")
    plt.tight_layout()
    plt.show()

## Comparison: Fixed Hyperparams (Notebook 04) vs Optuna-Tuned

Load the quick-stop results from notebook 04 (fixed defaults) and compare them
side-by-side with the Optuna-tuned results from this notebook.

In [ ]:
# Load notebook 04 results (fixed hyperparams) from results/gpt-4o/quick-stop/
FIXED_RESULTS_ROOTS = {
    s: str(repo_root / "results" / PSEUDO_LABEL_SOURCE / "quick-stop" / s)
    for s in STRATEGIES
}

fixed_lookup = {}  # strategy -> event -> result dict
for strategy in STRATEGIES:
    fixed_lookup[strategy] = {}
    for event in TARGET_EVENTS:
        for budget in RUN_BUDGETS:
            for seed_set in RUN_SEEDS:
                path = Path(FIXED_RESULTS_ROOTS[strategy]) / event / f"{budget}_set{seed_set}" / "metrics.json"
                if path.exists():
                    with open(path) as f:
                        fixed_lookup[strategy][event] = json.load(f)

# ---- Side-by-side comparison for each strategy ----
print("Fixed vs Optuna: Test Macro-F1  (budget=50, seed=1)")
print("=" * 90)

for strategy in STRATEGIES:
    print(f"\nStrategy: {strategy}")
    print(f"{'Event':<35} {'Fixed':>7} {'Optuna':>7} {'Delta':>8}")
    print("-" * 60)

    fixed_vals = []
    optuna_vals = []
    deltas = []

    for event in TARGET_EVENTS:
        f_r = fixed_lookup.get(strategy, {}).get(event)
        o_r = lookup.get(strategy, {}).get(event)
        f_f1 = f_r["test_macro_f1"] if f_r else None
        o_f1 = o_r["test_macro_f1"] if o_r else None

        f_str = f"{f_f1:.4f}" if f_f1 is not None else "N/A"
        o_str = f"{o_f1:.4f}" if o_f1 is not None else "N/A"

        if f_f1 is not None and o_f1 is not None:
            d = o_f1 - f_f1
            deltas.append(d)
            fixed_vals.append(f_f1)
            optuna_vals.append(o_f1)
            sign = "+" if d >= 0 else ""
            d_str = f"{sign}{d:.4f}"
        else:
            d_str = "N/A"

        print(f"{event:<35} {f_str:>7} {o_str:>7} {d_str:>8}")

    if deltas:
        mean_d = sum(deltas) / len(deltas)
        mean_f = sum(fixed_vals) / len(fixed_vals)
        mean_o = sum(optuna_vals) / len(optuna_vals)
        sign = "+" if mean_d >= 0 else ""
        print(f"{'MEAN':<35} {mean_f:>7.4f} {mean_o:>7.4f} {sign}{mean_d:>7.4f}")

# ---- Grouped bar chart: Fixed vs Optuna for baseline strategy ----
print("\n")

fig, ax = plt.subplots(figsize=(max(12, len(TARGET_EVENTS) * 1.2), 5))
x = np.arange(len(TARGET_EVENTS))
w = 0.35

fixed_f1s = [
    fixed_lookup.get("baseline", {}).get(e, {}).get("test_macro_f1", 0)
    if isinstance(fixed_lookup.get("baseline", {}).get(e), dict) else 0
    for e in TARGET_EVENTS
]
optuna_f1s = [
    lookup.get("baseline", {}).get(e, {}).get("test_macro_f1", 0)
    if isinstance(lookup.get("baseline", {}).get(e), dict) else 0
    for e in TARGET_EVENTS
]

ax.bar(x - w / 2, fixed_f1s,  w, label="Fixed hyperparams (NB 04)", color="steelblue", alpha=0.85)
ax.bar(x + w / 2, optuna_f1s, w, label="Optuna-tuned (NB 09)",     color="darkorange", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([e.replace("_", "\n") for e in TARGET_EVENTS], fontsize=8)
ax.set_ylabel("Test Macro-F1")
ax.set_ylim(0, 1)
ax.set_title(
    f"Baseline Stopping: Fixed vs Optuna Hyperparams -- Budget=50, Seed=1\n"
    f"(pseudo-labels: {PSEUDO_LABEL_SOURCE})",
    fontsize=11,
)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rebuild multi-tab dashboard so all quick-run result sets appear as tabs.

from lg_cotrain.dashboard import discover_result_sets, generate_html_multi

TOP_RESULTS_ROOT = str(repo_root / "results")

result_sets = discover_result_sets(TOP_RESULTS_ROOT)
print(f"Discovered {len(result_sets)} model(s):")
for model, types in result_sets.items():
    for exp_type, experiments in types.items():
        for name, path in experiments:
            print(f"  {model}/{exp_type}/{name:<20} -> {path}")

html = generate_html_multi(result_sets, data_root=DATA_ROOT)
dashboard_path = Path(TOP_RESULTS_ROOT) / "dashboard.html"
dashboard_path.write_text(html, encoding="utf-8")
print(f"\nDashboard written to: {dashboard_path}")
print("Open in a browser to compare strategies side-by-side.")